# DNN Improved — Parent Genre Classification
**DS340 — Techniques from [Music Genre Classification Using CNNs](https://github.com/crlandsc/Music-Genre-Classification-Using-Convolutional-Neural-Networks)**

Improvements over baseline NN-C (42.3% accuracy):
1. **BatchNorm** after each linear layer
2. **Gaussian noise augmentation** (tabular analog of spectrogram flipping)
3. **ReduceLROnPlateau** scheduler
4. **Wider architecture**: 512 → 256 → 128 → 64, dropout=0.4, 400 epochs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 1. Load & Preprocess

In [ ]:
GENRE_MAP = {
    'rock': 'Rock', 'alt-rock': 'Rock', 'punk-rock': 'Rock',
    'hard-rock': 'Rock', 'psych-rock': 'Rock', 'grunge': 'Rock',
    'rock-n-roll': 'Rock', 'emo': 'Rock', 'indie': 'Rock',
    'alternative': 'Rock', 'rockabilly': 'Rock', 'punk': 'Rock',
    'metal': 'Metal', 'heavy-metal': 'Metal', 'death-metal': 'Metal',
    'black-metal': 'Metal', 'metalcore': 'Metal', 'grindcore': 'Metal',
    'hardcore': 'Metal', 'goth': 'Metal', 'industrial': 'Metal',
    'edm': 'Electronic', 'electronic': 'Electronic', 'techno': 'Electronic',
    'detroit-techno': 'Electronic', 'minimal-techno': 'Electronic',
    'trance': 'Electronic', 'dubstep': 'Electronic', 'drum-and-bass': 'Electronic',
    'idm': 'Electronic', 'electro': 'Electronic', 'breakbeat': 'Electronic',
    'hardstyle': 'Electronic', 'ambient': 'Electronic',
    'house': 'House/Dance', 'deep-house': 'House/Dance',
    'chicago-house': 'House/Dance', 'progressive-house': 'House/Dance',
    'dance': 'House/Dance', 'disco': 'House/Dance', 'club': 'House/Dance',
    'garage': 'House/Dance', 'dancehall': 'House/Dance',
    'pop': 'Pop', 'indie-pop': 'Pop', 'synth-pop': 'Pop',
    'power-pop': 'Pop', 'k-pop': 'Pop', 'j-pop': 'Pop',
    'cantopop': 'Pop', 'mandopop': 'Pop', 'j-idol': 'Pop',
    'j-dance': 'Pop', 'party': 'Pop', 'happy': 'Pop',
    'pop-film': 'Pop', 'disney': 'Pop', 'show-tunes': 'Pop',
    'hip-hop': 'Hip-Hop/R&B', 'r-n-b': 'Hip-Hop/R&B', 'soul': 'Hip-Hop/R&B',
    'funk': 'Hip-Hop/R&B', 'groove': 'Hip-Hop/R&B', 'trip-hop': 'Hip-Hop/R&B',
    'latin': 'Latin', 'latino': 'Latin', 'salsa': 'Latin',
    'samba': 'Latin', 'reggaeton': 'Latin', 'tango': 'Latin',
    'forro': 'Latin', 'pagode': 'Latin', 'sertanejo': 'Latin',
    'mpb': 'Latin', 'brazil': 'Latin', 'bossanova': 'Latin',
    'romance': 'Latin', 'spanish': 'Latin',
    'jazz': 'Jazz/Blues', 'blues': 'Jazz/Blues', 'gospel': 'Jazz/Blues',
    'classical': 'Classical/Instrumental', 'opera': 'Classical/Instrumental',
    'piano': 'Classical/Instrumental', 'guitar': 'Classical/Instrumental',
    'new-age': 'Classical/Instrumental', 'sleep': 'Classical/Instrumental',
    'study': 'Classical/Instrumental',
    'country': 'Country/Folk', 'folk': 'Country/Folk', 'bluegrass': 'Country/Folk',
    'honky-tonk': 'Country/Folk', 'singer-songwriter': 'Country/Folk',
    'songwriter': 'Country/Folk', 'acoustic': 'Country/Folk',
    'reggae': 'Reggae', 'dub': 'Reggae', 'ska': 'Reggae',
    'world-music': 'World/Other', 'afrobeat': 'World/Other',
    'indian': 'World/Other', 'iranian': 'World/Other',
    'turkish': 'World/Other', 'malay': 'World/Other',
    'french': 'World/Other', 'german': 'World/Other',
    'swedish': 'World/Other', 'british': 'World/Other',
    'j-rock': 'World/Other', 'anime': 'World/Other',
    'children': 'World/Other', 'kids': 'World/Other',
    'comedy': 'World/Other', 'sad': 'World/Other',
    'chill': 'World/Other',
}

data = pd.read_csv('../Data/spotify-tracks-dataset.csv')
drop_cols = ['Unnamed: 0.1', 'Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name']
data = data.drop(columns=[c for c in drop_cols if c in data.columns])
data = data.dropna().drop_duplicates()

data['parent_genre'] = data['track_genre'].map(GENRE_MAP).fillna('World/Other')
data = data.groupby('parent_genre').sample(n=1000, random_state=42)
print(f'Shape: {data.shape}')
print(data['parent_genre'].value_counts())

In [ ]:
feature_cols = [
    'popularity', 'duration_ms', 'explicit',
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature'
]

le = LabelEncoder()
data['label'] = le.fit_transform(data['parent_genre'])
num_classes = len(le.classes_)

X = data[feature_cols].copy()
X['explicit'] = X['explicit'].astype(int)
y = data['label'].values

del data
gc.collect()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

del X, X_temp, y_temp
gc.collect()

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_sc).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_t   = torch.FloatTensor(X_val_sc).to(device)
y_val_t   = torch.LongTensor(y_val).to(device)
X_test_t  = torch.FloatTensor(X_test_sc).to(device)
y_test_t  = torch.LongTensor(y_test).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

print(f'Classes: {num_classes} — {list(le.classes_)}')
print(f'Train: {X_train_sc.shape[0]:,}  Val: {X_val_sc.shape[0]:,}  Test: {X_test_sc.shape[0]:,}')

---
## 2. Model Definition

In [ ]:
class ImprovedGenreClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims, dropout=0.4):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_model(name, hidden_dims, epochs=400, patience=30, lr=0.001,
                noise_std=0.05, dropout=0.4):
    model = ImprovedGenreClassifier(
        input_dim=X_train_sc.shape[1],
        num_classes=num_classes,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # ReduceLROnPlateau — halve LR when val loss stalls for 10 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    best_val_loss = float('inf')
    best_state = None
    counter = 0
    train_losses, val_losses, val_accs = [], [], []

    for epoch in range(epochs):
        # train with Gaussian noise augmentation
        model.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            xb_aug = xb + torch.randn_like(xb) * noise_std
            optimizer.zero_grad()
            loss = criterion(model(xb_aug), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        train_losses.append(epoch_loss / len(X_train_t))

        # validate
        model.eval()
        with torch.no_grad():
            out = model(X_val_t)
            val_loss = criterion(out, y_val_t).item()
            val_acc  = (out.argmax(1) == y_val_t).float().mean().item()
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f'  Early stop at epoch {epoch+1}')
                break

        if (epoch + 1) % 50 == 0:
            print(f'  Epoch {epoch+1:3d} | Train: {train_losses[-1]:.4f} | '
                  f'Val: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
                  f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        preds = model(X_val_t).argmax(1).cpu().numpy()

    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds, average='macro')
    print(f'\n{name} — Val Accuracy: {acc:.4f}, Macro F1: {f1:.4f}')

    return {
        'name': name, 'model': model,
        'accuracy': acc, 'macro_f1': f1, 'preds': preds,
        'train_losses': train_losses, 'val_losses': val_losses, 'val_accs': val_accs,
    }

print('Model defined.')

---
## 3. Train NN-D (Improved)

In [ ]:
print('=== NN-D: 512 → 256 → 128 → 64 | BatchNorm + Noise Aug + LR Scheduler ===')
nnd = train_model('NN-D (Improved)', hidden_dims=[512, 256, 128, 64])
gc.collect()

---
## 4. Results vs Baseline

In [ ]:
results = pd.DataFrame([
    {'Model': 'Random Forest (114 genres)',       'Accuracy': 0.3019, 'Macro F1': 0.2834},
    {'Model': 'NN-A — 1 layer (baseline)',         'Accuracy': 0.3856, 'Macro F1': 0.3669},
    {'Model': 'NN-B — 2 layers (baseline)',        'Accuracy': 0.4167, 'Macro F1': 0.4017},
    {'Model': 'NN-C — 3 layers (baseline)',        'Accuracy': 0.4233, 'Macro F1': 0.4120},
    {'Model': nnd['name'], 'Accuracy': nnd['accuracy'], 'Macro F1': nnd['macro_f1']},
])

print('=== All Models — Validation Set ===')
print(results.to_string(index=False))
print(f"\nImprovement over NN-C: {nnd['accuracy'] - 0.4233:+.4f} accuracy")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results))
w = 0.35
ax.bar(x - w/2, results['Accuracy'], w, label='Accuracy')
ax.bar(x + w/2, results['Macro F1'], w, label='Macro F1')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Baseline DNNs vs Improved NN-D')
ax.axvline(x=3.5, color='gray', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Training Curve & Classification Report

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nnd['train_losses'], label='Train Loss')
ax.plot(nnd['val_losses'], label='Val Loss')
ax2 = ax.twinx()
ax2.plot(nnd['val_accs'], color='green', linestyle='--', label='Val Acc')
ax2.set_ylabel('Val Accuracy', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax.set_title('NN-D: BatchNorm + Noise Augmentation + LR Scheduling')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(classification_report(y_val, nnd['preds'], target_names=le.classes_))

---
## 6. Test Set — Final Score
Run only once, after all tuning is done.

In [ ]:
nnd['model'].eval()
with torch.no_grad():
    test_preds = nnd['model'](X_test_t).argmax(1).cpu().numpy()

print(f"NN-D (Improved) — TEST SET")
print(f"Accuracy : {accuracy_score(y_test, test_preds):.4f}")
print(f"Macro F1 : {f1_score(y_test, test_preds, average='macro'):.4f}")

## AI Usage
- Used Claude to implement BatchNorm, noise augmentation, and LR scheduling based on reference CNN paper
- Reference: [Music Genre Classification Using CNNs](https://github.com/crlandsc/Music-Genre-Classification-Using-Convolutional-Neural-Networks)